**<center><h1>Docling for Data Extraction: Testing Its Performance in Table Extraction</h1></center>**

In [1]:
%%capture
#! pip install -U ipywidgets
! pip install docling
! pip install pdf2image
! apt-get update && apt-get install -y poppler-utils

In [2]:
import logging
import time
from pathlib import Path
import pandas as pd
from pdf2image import convert_from_path
import matplotlib.pyplot as plt
from PIL import Image
import os
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode

Also let's define another function for data extraction with Docling.

In [ ]:
def extract_data_with_docling(input_data_path):
    """
    Extracts data from a PDF or image file using the Docling library. 
    Displays the document's content as markdown and exports any tables found in the document to CSV files.

    Args:
        input_data_path (str): The path to the input file (PDF or image).
    """
    
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = False
    pipeline_options.do_table_structure = True
    pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE

    # Create a document converter with specified format options
    doc_converter = DocumentConverter(
        allowed_formats=[
                InputFormat.PDF,
                InputFormat.IMAGE,
            ],  # whitelist formats, non-matching files are ignored.
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    result = doc_converter.convert(input_data_path)
    print(result.document.export_to_markdown())

    doc_filename = result.input.file.stem
    # Loop through each table detected in the document and Export it
    for table_ix, table in enumerate(result.document.tables):
        table_df: pd.DataFrame = table.export_to_dataframe()
        print(f"## Table {table_ix}")
        #print(table_df.to_markdown())

        # Save the table as csv
        element_csv_filename = f"{doc_filename}-table-{table_ix+1}.csv"
        table_df.to_csv(element_csv_filename)